# Fake News & Misinformation Detection — Data Science + Baseline ML

**Phase 1 deliverable:** dataset audit, cleaning, exploratory data analysis, linguistic feature engineering, statistical comparison, and a reproducible TF-IDF + Logistic Regression baseline.

**Dataset files:** `Fake.csv` and `True.csv` from the uploaded project. Expected columns: `title`, `text`, `subject`, `date`.

**Label convention:** `0 = Fake`, `1 = Real`.

> **Important limitations**
> - These labels represent the supplied dataset, not independently verified truth for every claim.
> - The dataset does not contain a publisher/source identifier. A defensible source-credibility score cannot be computed from these files alone.
> - Dataset-specific patterns (including topics, formatting, and time period) can make classification look stronger than real-world performance. Interpret metrics cautiously and test on external, newer data before deployment.
> - Evidence retrieval and DistilBERT fine-tuning are later phases; this notebook establishes the EDA and required baseline first.

## 1. Environment and reproducibility

In [ ]:
import os, re, string, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

print("Python environment ready.")

## 2. Load the two CSV datasets
In Google Colab, upload both files when prompted if they are not already in `/content`. In a local Jupyter environment, place them beside this notebook.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
fake_path = DATA_DIR / "Fake.csv"
true_path = DATA_DIR / "True.csv"

if not (fake_path.exists() and true_path.exists()):
    try:
        from google.colab import files
        print("Upload Fake.csv and True.csv when the file picker opens.")
        uploaded = files.upload()
        # Uploaded files are normally saved in the current working directory.
        cwd = Path.cwd()
        fake_path = cwd / "Fake.csv"
        true_path = cwd / "True.csv"
    except ImportError:
        pass

if not fake_path.exists() or not true_path.exists():
    raise FileNotFoundError(
        "Could not find Fake.csv and True.csv. Place both files in the notebook's "
        "working directory or upload them in Colab."
    )

fake_raw = pd.read_csv(fake_path, low_memory=False)
true_raw = pd.read_csv(true_path, low_memory=False)

print("Fake dataset:", fake_raw.shape)
print("Real dataset:", true_raw.shape)
display(fake_raw.head(3))
display(true_raw.head(3))

## 3. Schema and data-quality audit

In [ ]:
required_columns = {"title", "text"}
for name, df in [("Fake", fake_raw), ("Real", true_raw)]:
    missing_required = required_columns - set(df.columns)
    if missing_required:
        raise ValueError(f"{name} dataset is missing required columns: {missing_required}")

schema_report = pd.DataFrame({
    "Fake_dtype": fake_raw.dtypes.astype(str),
    "Real_dtype": true_raw.dtypes.astype(str),
    "Fake_missing": fake_raw.isna().sum(),
    "Real_missing": true_raw.isna().sum(),
})
display(schema_report)

quality_before = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Exact duplicate rows": int(df.duplicated().sum()),
        "Missing title": int(df["title"].isna().sum()),
        "Missing text": int(df["text"].isna().sum()),
        "Blank title": int(df["title"].fillna("").astype(str).str.strip().eq("").sum()),
        "Blank text": int(df["text"].fillna("").astype(str).str.strip().eq("").sum()),
    }
    for name, df in [("Fake", fake_raw), ("Real", true_raw)]
])
display(quality_before)

## 4. Clean and combine data
The original datasets are preserved as `fake_raw` and `true_raw`. Cleaning removes exact duplicate records within each class, drops rows without usable article text, and trims whitespace. Cross-class duplicates are audited separately because conflicting labels may indicate data-quality issues.

In [ ]:
def clean_dataset(df):
    df = df.copy()
    for col in ["title", "text", "subject", "date"]:
        if col in df.columns:
            df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    df = df.drop_duplicates()
    df = df[df["text"].notna()]
    df = df[df["text"].astype(str).str.strip().ne("")]
    return df.reset_index(drop=True)

fake_clean = clean_dataset(fake_raw)
true_clean = clean_dataset(true_raw)
fake_clean["label"] = 0
true_clean["label"] = 1

news_df = pd.concat([fake_clean, true_clean], ignore_index=True)
news_df = news_df.drop_duplicates().reset_index(drop=True)

print("Clean fake rows:", len(fake_clean))
print("Clean real rows:", len(true_clean))
print("Combined rows after exact duplicate removal:", len(news_df))
display(news_df.head())
display(news_df["label"].map({0:"Fake", 1:"Real"}).value_counts().to_frame("Article count"))

In [ ]:
# Check exact article-text duplicates that carry conflicting labels.
text_key = news_df["text"].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
text_label_counts = pd.DataFrame({"text_key": text_key, "label": news_df["label"]}).groupby("text_key")["label"].nunique()
conflicting_text_keys = text_label_counts[text_label_counts > 1]
print("Normalized article texts appearing under both labels:", len(conflicting_text_keys))
print("Rows associated with those conflicting texts:",
      int(text_key.isin(conflicting_text_keys.index).sum()))
print("These should be reviewed; this notebook does not silently relabel them.")

## 5. Linguistic feature engineering
These descriptive features support EDA. They are not used as extra model inputs in the baseline, which remains a TF-IDF text model.

In [ ]:
def count_words(text):
    return len(re.findall(r"\b\w+\b", str(text)))

def count_sentences(text):
    return len([s for s in re.split(r"[.!?]+", str(text)) if s.strip()])

def average_word_length(text):
    words = re.findall(r"\b[a-zA-Z]+\b", str(text))
    return float(np.mean([len(w) for w in words])) if words else 0.0

def unique_word_count(text):
    return len(set(re.findall(r"\b\w+\b", str(text).lower())))

def punctuation_count(text):
    return sum(ch in string.punctuation for ch in str(text))

def uppercase_count(text):
    return sum(ch.isupper() for ch in str(text))

def uppercase_ratio(text):
    letters = [ch for ch in str(text) if ch.isalpha()]
    return sum(ch.isupper() for ch in letters) / len(letters) if letters else 0.0

# Use article text for body-level features; include title separately in model input.
news_df["word_count"] = news_df["text"].map(count_words)
news_df["sentence_count"] = news_df["text"].map(count_sentences)
news_df["character_count"] = news_df["text"].astype(str).str.len()
news_df["average_word_length"] = news_df["text"].map(average_word_length)
news_df["unique_word_count"] = news_df["text"].map(unique_word_count)
news_df["punctuation_count"] = news_df["text"].map(punctuation_count)
news_df["uppercase_count"] = news_df["text"].map(uppercase_count)
news_df["uppercase_ratio"] = news_df["text"].map(uppercase_ratio)
news_df["unique_word_ratio"] = (
    news_df["unique_word_count"] / news_df["word_count"].replace(0, np.nan)
).fillna(0)
news_df["average_sentence_length"] = (
    news_df["word_count"] / news_df["sentence_count"].replace(0, np.nan)
).fillna(0)

feature_columns = [
    "word_count", "sentence_count", "character_count", "average_word_length",
    "unique_word_count", "punctuation_count", "uppercase_count",
    "uppercase_ratio", "unique_word_ratio", "average_sentence_length"
]
display(news_df[feature_columns].describe().T.round(3))

## 6. Exploratory data analysis (EDA)

In [ ]:
class_counts = news_df["label"].map({0: "Fake", 1: "Real"}).value_counts()
display(class_counts.to_frame("Article count"))

plt.figure(figsize=(7, 4))
sns.countplot(data=news_df, x="label", hue="label", palette="Set2", legend=False)
plt.xticks([0, 1], ["Fake", "Real"])
plt.title("Class distribution")
plt.xlabel("Label")
plt.ylabel("Articles")
plt.tight_layout()
plt.show()

print("Class proportions:")
display(news_df["label"].value_counts(normalize=True).rename(index={0:"Fake",1:"Real"}).mul(100).round(2).to_frame("Percent"))

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=news_df, x="word_count", hue="label", bins=60,
             element="step", stat="density", common_norm=False)
plt.xlim(0, news_df["word_count"].quantile(0.99))
plt.title("Article word-count distribution (99th percentile shown)")
plt.xlabel("Words in article")
plt.tight_layout()
plt.show()

features_to_plot = ["word_count", "sentence_count", "average_word_length",
                    "unique_word_ratio", "punctuation_count", "uppercase_ratio"]
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for feature, ax in zip(features_to_plot, axes.flatten()):
    sns.boxplot(data=news_df, x="label", y=feature, hue="label",
                palette="Set2", legend=False, showfliers=False, ax=ax)
    ax.set_xticks([0, 1], ["Fake", "Real"])
    ax.set_title(feature.replace("_", " ").title())
plt.tight_layout()
plt.show()

In [ ]:
feature_comparison = news_df.groupby("label")[feature_columns].agg(["mean", "median", "std"]).round(3)
feature_comparison.columns = [f"{stat}_{feature}" for feature, stat in feature_comparison.columns]
feature_comparison.index = feature_comparison.index.map({0:"Fake", 1:"Real"})
display(feature_comparison.T)

In [ ]:
if "subject" in news_df.columns:
    subject_summary = pd.crosstab(news_df["subject"].fillna("Unknown"),
                                  news_df["label"].map({0:"Fake", 1:"Real"}))
    display(subject_summary.sort_values(by=list(subject_summary.columns), ascending=False))
    subject_summary.plot(kind="bar", stacked=False, figsize=(12, 5))
    plt.title("Article counts by dataset subject and label")
    plt.ylabel("Articles")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No subject column exists in the supplied data.")

## 7. Date coverage and topic/trend caveats
The `date` field is parsed defensively because source data can contain inconsistent date strings. These charts describe the dataset's coverage and label distribution, not live misinformation trends.

In [ ]:
if "date" in news_df.columns:
    news_df["date_parsed"] = pd.to_datetime(news_df["date"], errors="coerce", utc=True)
    print("Dates parsed:", int(news_df["date_parsed"].notna().sum()), "/", len(news_df))
    dated = news_df.dropna(subset=["date_parsed"]).copy()
    if not dated.empty:
        dated["month"] = dated["date_parsed"].dt.to_period("M").astype(str)
        monthly = pd.crosstab(dated["month"], dated["label"].map({0:"Fake",1:"Real"})).sort_index()
        display(monthly.tail(20))
        monthly.tail(36).plot(figsize=(13, 5))
        plt.title("Dataset article counts by month (last 36 observed months)")
        plt.ylabel("Articles")
        plt.xlabel("Month")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("No date column exists.")

## 8. Statistical test: linguistic feature differences
Welch's t-test compares means without assuming equal variances. Because many features are tested, Benjamini–Hochberg false-discovery-rate correction is applied. Statistical significance does not imply a feature is useful for real-world classification or that emotional language proves misinformation.

In [ ]:
from scipy.stats import ttest_ind

def benjamini_hochberg(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted_ranked = ranked * n / np.arange(1, n + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted = np.empty(n)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted

test_rows = []
for feature in feature_columns:
    fake_vals = news_df.loc[news_df["label"] == 0, feature].dropna()
    real_vals = news_df.loc[news_df["label"] == 1, feature].dropna()
    stat, p = ttest_ind(fake_vals, real_vals, equal_var=False, nan_policy="omit")
    test_rows.append({
        "feature": feature,
        "fake_mean": fake_vals.mean(),
        "real_mean": real_vals.mean(),
        "mean_difference_fake_minus_real": fake_vals.mean() - real_vals.mean(),
        "t_statistic": stat,
        "p_value": p
    })
tests_df = pd.DataFrame(test_rows)
tests_df["p_value_fdr_bh"] = benjamini_hochberg(tests_df["p_value"].fillna(1))
tests_df["significant_fdr_0_05"] = tests_df["p_value_fdr_bh"] < 0.05
display(tests_df.sort_values("p_value_fdr_bh").round(6))

## 9. Baseline model: TF-IDF + Logistic Regression
The model uses title + body text, a stratified train/test split, and a scikit-learn Pipeline so TF-IDF vocabulary/IDF are fitted only on training data. The held-out test set is not used for tuning. Scores are internal benchmark results and can be inflated by dataset-specific artifacts or duplicated/reposted stories.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)

# Remove normalized exact body-text duplicates across labels from model evaluation,
# because conflicting copies make evaluation ambiguous.
model_df = news_df.copy()
model_df["_text_key"] = model_df["text"].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
conflict_keys = set(conflicting_text_keys.index)
model_df = model_df[~model_df["_text_key"].isin(conflict_keys)].copy()

# Use title and article body. Missing titles are handled safely.
model_df["model_text"] = (
    model_df["title"].fillna("").astype(str).str.strip() + " " +
    model_df["text"].fillna("").astype(str).str.strip()
).str.strip()
model_df = model_df[model_df["model_text"].ne("")].copy()

X = model_df["model_text"]
y = model_df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training class proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename(index={0:"Fake",1:"Real"}).to_frame("proportion"))

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=200_000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    ))
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

baseline_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (Real=1)": precision_score(y_test, y_pred, zero_division=0),
    "Recall (Real=1)": recall_score(y_test, y_pred, zero_division=0),
    "F1 (Real=1)": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}]).T.rename(columns={0:"Score"})
display(baseline_metrics.round(4))
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Fake", "Predicted Real"],
            yticklabels=["Actual Fake", "Actual Real"])
plt.title("Baseline confusion matrix")
plt.tight_layout()
plt.show()

## 10. Explainability: influential words and phrases
For Logistic Regression, coefficients indicate which TF-IDF terms are associated with each class in this fitted model. They are model associations, not evidence that a story is true or false.

In [ ]:
tfidf = baseline_model.named_steps["tfidf"]
lr = baseline_model.named_steps["classifier"]
terms = np.asarray(tfidf.get_feature_names_out())
coefficients = lr.coef_[0]

top_real_idx = np.argsort(coefficients)[-20:][::-1]
top_fake_idx = np.argsort(coefficients)[:20]

term_importance = pd.concat([
    pd.DataFrame({"Term": terms[top_fake_idx], "Coefficient": coefficients[top_fake_idx], "Association": "Fake"}),
    pd.DataFrame({"Term": terms[top_real_idx], "Coefficient": coefficients[top_real_idx], "Association": "Real"})
], ignore_index=True)
display(term_importance)

## 11. Error analysis
Inspect false positives and false negatives to understand failure modes. Avoid interpreting a model mistake as a definitive truth assessment.

In [ ]:
error_df = pd.DataFrame({
    "title": model_df.loc[X_test.index, "title"].fillna("").values,
    "text": X_test.values,
    "actual_label": y_test.values,
    "predicted_label": y_pred,
    "probability_real": y_prob
})
error_df["actual_label"] = error_df["actual_label"].map({0:"Fake",1:"Real"})
error_df["predicted_label"] = error_df["predicted_label"].map({0:"Fake",1:"Real"})
errors = error_df[error_df["actual_label"] != error_df["predicted_label"]].copy()
print("Misclassified test examples:", len(errors), "of", len(error_df))
display(errors[["title", "actual_label", "predicted_label", "probability_real"]].head(20))

## 12. Save project handoff artifacts
Files are saved to the notebook's current working directory. In Colab, use the Files panel to download them or run the optional download cell below.

In [ ]:
OUTPUT_DIR = Path.cwd()
feature_export_cols = [c for c in [
    "title", "text", "subject", "date", "label", *feature_columns
] if c in news_df.columns]

news_df[feature_export_cols].to_csv(OUTPUT_DIR / "fake_news_linguistic_features.csv", index=False)
feature_comparison.to_csv(OUTPUT_DIR / "fake_news_feature_comparison.csv")
tests_df.to_csv(OUTPUT_DIR / "fake_news_statistical_tests.csv", index=False)
baseline_metrics.to_csv(OUTPUT_DIR / "fake_news_baseline_metrics.csv")
term_importance.to_csv(OUTPUT_DIR / "fake_news_baseline_term_importance.csv", index=False)
errors[["title", "actual_label", "predicted_label", "probability_real"]].to_csv(
    OUTPUT_DIR / "fake_news_baseline_errors.csv", index=False
)

handoff_df = news_df[feature_export_cols].copy()
handoff_df.to_csv(OUTPUT_DIR / "fake_news_DS_DA_handoff.csv", index=False)

quality_report = pd.DataFrame([
    ["Raw fake rows", len(fake_raw)],
    ["Raw real rows", len(true_raw)],
    ["Cleaned combined rows", len(news_df)],
    ["Fake rows", int((news_df["label"] == 0).sum())],
    ["Real rows", int((news_df["label"] == 1).sum())],
    ["Missing article text", int(news_df["text"].isna().sum())],
    ["Exact duplicate rows after cleaning", int(news_df.duplicated().sum())],
    ["Cross-label conflicting normalized texts", len(conflicting_text_keys)],
    ["Rows used in baseline", len(model_df)],
    ["Baseline training rows", len(X_train)],
    ["Baseline test rows", len(X_test)],
], columns=["Metric", "Value"])
display(quality_report)
quality_report.to_csv(OUTPUT_DIR / "fake_news_data_quality_report.csv", index=False)
print("Saved CSV handoff files to:", OUTPUT_DIR)

## 13. Interpretation and next phase

### What this notebook delivers
- Data-quality audit and cleaning
- Linguistic feature engineering and EDA
- Subject and date exploration where columns support it
- Welch tests with multiple-testing correction
- Required TF-IDF + Logistic Regression baseline
- Classification metrics, confusion matrix, coefficient-based term inspection, and error analysis
- CSV artifacts for downstream application development

### What cannot be concluded from these files alone
- **Publisher credibility:** neither file has a publisher/source column. Do not treat `subject` as publisher identity. To build a source credibility index, obtain publisher-level provenance and independently adjudicated historical labels.
- **Live misinformation trends:** the dataset is historical and may be biased; it does not represent real-time news volume.
- **Evidence support/contradiction:** nearest-neighbour similarity alone cannot establish that a retrieved article supports or refutes a claim. The later evidence module needs traceable article URLs, dates, publisher identity, and ideally a stance/entailment assessment.
- **Truth guarantee:** classifier output is a model estimate, not a fact-check.

### Recommended next ML milestones (after review)
1. Compare this baseline with a controlled evaluation that groups duplicate/near-duplicate stories to reduce leakage.
2. Add a DistilBERT model and compare F1, ROC-AUC, training time, model size, and inference latency.
3. Build a semantic retrieval index from traceable article records and evaluate relevance/stance separately.
4. Design source credibility scoring only after acquiring reliable source-level ground truth.
5. Package inference into an API/application only after the notebook and model evaluation are approved.